In [41]:
import pyxdf
import pandas as pd
import numpy as np
from datetime import datetime
import os
import glob

In [42]:
def export_streams_to_csv(streams, output_dir="exported_streams"):

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    exported_files = []
    for idx, stream in enumerate(streams):
        # Get stream metadata
        stream_name = stream['info']['name'][0]
        source_id = stream['info'].get('source_id', ['Unknown'])[0]
        if source_id is None:
            source_id = "Unknown"

        if source_id == "MD-V5-0001023":
            filename = f"{stream_name}_C.csv"
        elif source_id == "MD-V6-0000405":
            filename = f"{stream_name}_I.csv"
        else:
            filename = f"{stream_name}.csv"
        filepath = os.path.join(output_dir, filename)

        # Get data and timestamps
        data = stream['time_series']
        timestamps = stream['time_stamps']

        # Skip streams with no data
        if len(data) == 0:
            print(f"Skipping stream {idx} ({stream_name}): No data")
            continue

        # Convert data to numpy array if it's not already
        if not isinstance(data, np.ndarray):
            data = np.array(data)

        # Special handling for Neon Companion stream - export all channels with proper labels
        if "Neon" in stream_name and "Companion" in stream_name:
            # Get all channel labels
            channel_labels = []
            try:
                desc = stream['info'].get('desc')
                if desc and len(desc) > 0 and desc[0] is not None and isinstance(desc[0], dict):
                    channels_info = desc[0].get('channels')
                    if channels_info and len(channels_info) > 0 and channels_info[0] is not None:
                        channel_list = channels_info[0].get('channel', [])
                        for i, ch in enumerate(channel_list):
                            if isinstance(ch, dict) and 'label' in ch:
                                label = ch['label'][0] if isinstance(ch['label'], list) else ch['label']
                                channel_labels.append(label)
                            else:
                                channel_labels.append(f'channel_{i}')
            except (AttributeError, KeyError, TypeError):
                # Fallback to generic channel names
                channel_labels = [f'channel_{i}' for i in range(data.shape[1])]

            # Create DataFrame with all channels and proper labels
            df_dict = {'timestamp': timestamps}
            for i, label in enumerate(channel_labels):
                if i < data.shape[1]:
                    df_dict[label] = data[:, i]
            
            df = pd.DataFrame(df_dict)
            print(f"Exported Neon stream with {len(channel_labels)} channels: {', '.join(channel_labels)}")

        # Regular handling for other streams
        elif data.ndim == 1:
            # Single channel data
            df = pd.DataFrame({
                'timestamp': timestamps,
                'value': data
            })
        else:
            # Multi-channel data
            channel_count = data.shape[1] if len(data.shape) > 1 else 1
            df_dict = {'timestamp': timestamps}

            if channel_count == 1:
                # Treat as single channel even if it's 2D with 1 column
                df_dict['value'] = data.flatten()
            else:
                # Get channel labels if available (with safer handling)
                channel_labels = []
                try:
                    desc = stream['info'].get('desc')
                    if desc and len(desc) > 0 and desc[0] is not None and isinstance(desc[0], dict):
                        channels_info = desc[0].get('channels')
                        if channels_info and len(channels_info) > 0 and channels_info[0] is not None:
                            channel_list = channels_info[0].get('channel', [])
                            if channel_list:
                                for i in range(channel_count):
                                    if i < len(channel_list):
                                        label = channel_list[i].get('label', [f'channel_{i}'])
                                        if isinstance(label, list):
                                            label = label[0] if label else f'channel_{i}'
                                        channel_labels.append(label)
                                    else:
                                        channel_labels.append(f'channel_{i}')
                            else:
                                channel_labels = [f'channel_{i}' for i in range(channel_count)]
                        else:
                            channel_labels = [f'channel_{i}' for i in range(channel_count)]
                    else:
                        channel_labels = [f'channel_{i}' for i in range(channel_count)]
                except (AttributeError, KeyError, TypeError):
                    channel_labels = [f'channel_{i}' for i in range(channel_count)]

                # Add channel data to DataFrame
                for i in range(channel_count):
                    df_dict[channel_labels[i]] = data[:, i]

            df = pd.DataFrame(df_dict)

        # Export to CSV
        df.to_csv(filepath, index=False)
        exported_files.append(filepath)

        print(f"Exported stream {idx} ({stream_name}) to {filepath}")
        print(f"  Source ID: {source_id}")
        print(f"  Data shape: {data.shape}")
        print(f"  Sample count: {len(timestamps)}")
        if "Neon" in stream_name:
            print(f"  First timestamp: {timestamps[0]}")
        print()

    print(f"Successfully exported {len(exported_files)} streams to {output_dir}/")
    return exported_files

In [43]:
def merge_csv_files(input_dir, output_path="merged_data.csv", use_ffill=True, tolerance=0.005, participant_id=None, condition=None, video_condition=None):
    """
    Merge CSV files using PPG_GRN_C as base, with nearest timestamp matching.

    Parameters:
    - input_dir: Directory containing CSV files
    - output_path: Path for the merged output file
    - use_ffill: Whether to forward fill missing values
    - tolerance: Maximum time difference for matching (seconds, default 0.005 = 5ms)
    - participant_id: ID to add as the first column in the output (optional)
    - condition: Condition to add as the second column in the output (optional)
    - video_condition: Video condition (1-6) for labeling DataSyncMarker sequences (optional)
    """
    import glob

    VIDEO_ORDERS = {
        1: ('Unreliable', 'Reliable', 'Mix'),
        2: ('Unreliable', 'Mix', 'Reliable'),
        3: ('Reliable', 'Unreliable', 'Mix'),
        4: ('Reliable', 'Mix', 'Unreliable'),
        5: ('Mix', 'Unreliable', 'Reliable'),
        6: ('Mix', 'Reliable', 'Unreliable')
    }

    csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
    if not csv_files:
        print(f"No CSV files found in {input_dir}")
        return None
    print(f"Found {len(csv_files)} CSV files to merge")

    ppg_base = None
    other_files = []
    datasync_file = None

    for file_path in csv_files:
        filename = os.path.basename(file_path)
        if "PPG_GRN" in filename and "_C.csv" in filename:
            ppg_base = file_path
            print(f"Found PPG_GRN base file: {filename}")
        elif "DataSyncMarker" in filename:
            datasync_file = file_path
        else:
            other_files.append(file_path)

    if ppg_base is None:
        print("No PPG_GRN_C.csv file found to use as base!")
        return None

    try:
        base_df = pd.read_csv(ppg_base)
        base_filename = os.path.basename(ppg_base)
        print(f"Base file: {base_filename} ({len(base_df)} rows)")
        base_df['timestamp'] = base_df['timestamp'].round(3)

        merged_df = pd.DataFrame()
        if participant_id is not None:
            merged_df['ID'] = [participant_id] * len(base_df)
        if condition is not None:
            merged_df['Condition'] = [condition] * len(base_df)
        merged_df['timestamp'] = base_df['timestamp']

        if datasync_file is not None:
            try:
                datasync_df = pd.read_csv(datasync_file)
                datasync_filename = os.path.basename(datasync_file)

                if 'timestamp' in datasync_df.columns:
                    print(f"\nProcessing {datasync_filename} (adding after timestamp)...")
                    datasync_df['timestamp'] = datasync_df['timestamp'].round(3)
                    print(f"  File timestamps: {datasync_df['timestamp'].min():.3f} to {datasync_df['timestamp'].max():.3f}")
                    data_columns = [col for col in datasync_df.columns if col != 'timestamp']
                    datasync_df = datasync_df.sort_values('timestamp').reset_index(drop=True)

                    if 'channel_0' in data_columns:
                        print(f"  Expanding DataSyncMarker events into time ranges...")
                        print(f"  Original DataSyncMarker events: {len(datasync_df)} events")
                        marker_events = datasync_df[['timestamp', 'channel_0']].sort_values('timestamp').copy()
                        print(f"  DataSyncMarker events:")
                        for idx, row in marker_events.iterrows():
                            print(f"    t={row['timestamp']:.3f}, value={row['channel_0']}")

                        merged_df = pd.merge_asof(merged_df,
                                                  marker_events[['timestamp', 'channel_0']],
                                                  on='timestamp',
                                                  direction='backward')

                        # Compute Reliability from raw channel_0 (1=video, 2=no video) before label transformation
                        if video_condition is not None and video_condition in VIDEO_ORDERS:
                            video_order = VIDEO_ORDERS[video_condition]
                            reliability_labels = []
                            video_segment_num = 0
                            prev_state = None
                            for state in merged_df['channel_0']:
                                state_int = int(state) if pd.notna(state) else None
                                if state_int != prev_state:
                                    if state_int == 1:
                                        video_segment_num += 1
                                    prev_state = state_int
                                if state_int == 1 and 1 <= video_segment_num <= len(video_order):
                                    reliability_labels.append(video_order[video_segment_num - 1])
                                elif state_int == 2:
                                    reliability_labels.append("No Video")
                                else:
                                    reliability_labels.append(None)
                            merged_df['Reliability'] = reliability_labels
                            print(f"  Added Reliability column using video condition {video_condition}: {video_order}")

                        if video_condition is not None:
                            print(f"  Applying video condition {video_condition} labeling...")
                            video_condition_labels = _label_video_condition(merged_df['channel_0'], video_condition)
                            merged_df = merged_df.rename(columns={'channel_0': 'DataSyncMarker'})
                            merged_df['DataSyncMarker'] = video_condition_labels
                        else:
                            merged_df = merged_df.rename(columns={'channel_0': 'DataSyncMarker_channel_0'})

                        print(f"  Added DataSyncMarker column after timestamp")
                        marker_col = 'DataSyncMarker' if video_condition is not None else 'DataSyncMarker_channel_0'
                        matched_count = merged_df[marker_col].notna().sum()
                        print(f"    {marker_col}: {matched_count}/{len(merged_df)} matches ({100*matched_count/len(merged_df):.1f}%)")

                        value_counts = merged_df[marker_col].value_counts()
                        print(f"  DataSyncMarker value distribution:")
                        for value, count in value_counts.items():
                            print(f"    {value}: {count} rows ({100*count/len(merged_df):.1f}%)")

                    remaining_columns = [col for col in data_columns if col != 'channel_0']
                    if remaining_columns:
                        remaining_df = datasync_df[['timestamp'] + remaining_columns].copy()
                        file_prefix = datasync_filename.replace('.csv', '').replace('stream_', '')
                        for col in remaining_columns:
                            remaining_df = remaining_df.rename(columns={col: f"{file_prefix}_{col}"})
                        temp_path = "temp_datasync_remaining.csv"
                        remaining_df.to_csv(temp_path, index=False)
                        other_files.append(temp_path)

            except Exception as e:
                print(f"Error processing DataSyncMarker file {datasync_file}: {str(e)}")

        base_prefix = base_filename.replace('.csv', '').replace('stream_', '')
        data_columns = [col for col in base_df.columns if col != 'timestamp']
        for col in data_columns:
            merged_df[f"{base_prefix}_{col}"] = base_df[col]
        merged_df = merged_df.sort_values('timestamp').reset_index(drop=True)

    except Exception as e:
        print(f"Error reading base file {ppg_base}: {str(e)}")
        return None

    print(f"Base PPG timestamps: {merged_df['timestamp'].min():.3f} to {merged_df['timestamp'].max():.3f}")

    for i, file_path in enumerate(other_files):
        try:
            filename = os.path.basename(file_path)
            if filename == "temp_datasync_remaining.csv":
                df = pd.read_csv(file_path)
                if os.path.exists(file_path):
                    os.remove(file_path)
            else:
                df = pd.read_csv(file_path)

            if 'timestamp' not in df.columns:
                print(f"Skipping {filename}: No 'timestamp' column found")
                continue

            df['timestamp'] = df['timestamp'].round(3)
            print(f"\nProcessing {filename}...")
            print(f"  File timestamps: {df['timestamp'].min():.3f} to {df['timestamp'].max():.3f}")

            if "Neon" in filename:
                pupil_columns = [col for col in df.columns if 'PupilDiameter' in col]
                if not pupil_columns:
                    print(f"Skipping {filename}: No PupilDiameter columns found")
                    continue
                print(f"  Downsampling PupilDiameter data from 200Hz to 25Hz using aggregation...")
                df = df[['timestamp'] + pupil_columns]
                min_time = df['timestamp'].min()
                max_time = df['timestamp'].max()
                bin_size = 0.04
                time_bins = np.round(np.arange(min_time, max_time + bin_size, bin_size), 3)
                df['time_bin'] = pd.cut(df['timestamp'], bins=time_bins, labels=time_bins[:-1], include_lowest=True)
                aggregated_data = []
                for bin_center in time_bins[:-1]:
                    bin_data = df[df['time_bin'] == bin_center]
                    if len(bin_data) > 0:
                        row_data = {'timestamp': bin_center}
                        for col in pupil_columns:
                            valid_values = bin_data[col].dropna()
                            row_data[col] = valid_values.mean() if len(valid_values) > 0 else np.nan
                        aggregated_data.append(row_data)
                df = pd.DataFrame(aggregated_data)
                print(f"  After aggregation: {len(df)} samples ({len(pupil_columns)} PupilDiameter columns)")
                for col in pupil_columns:
                    valid_count = df[col].notna().sum()
                    print(f"    {col}: {valid_count}/{len(df)} valid aggregated values ({100*valid_count/len(df):.1f}%)")

            data_columns = [col for col in df.columns if col != 'timestamp' and col != 'time_bin']
            df = df.sort_values('timestamp').reset_index(drop=True)
            df_for_merge = df.copy()

            if filename != "temp_datasync_remaining.csv":
                file_prefix = filename.replace('.csv', '').replace('stream_', '')
                for col in data_columns:
                    df_for_merge = df_for_merge.rename(columns={col: f"{file_prefix}_{col}"})

            # Track which columns are new after this merge
            cols_before = set(merged_df.columns)

            if filename == "temp_datasync_remaining.csv":
                print(f"  Expanding DataSyncMarker_channel_1 events into time ranges...")
                merged_df = pd.merge_asof(merged_df, df_for_merge, on='timestamp', direction='backward')
            elif "Keyboard" in filename:
                print(f"  Inserting {len(df)} Keyboard events at exact timestamps...")
                if participant_id is not None:
                    df_for_merge['ID'] = participant_id
                if condition is not None:
                    df_for_merge['Condition'] = condition
                merged_df = pd.concat([merged_df, df_for_merge], ignore_index=True).sort_values('timestamp').reset_index(drop=True)
            else:
                merged_df = pd.merge_asof(merged_df, df_for_merge, on='timestamp', direction='nearest', tolerance=tolerance)

            new_cols = [col for col in merged_df.columns if col not in cols_before]
            print(f"  Added {len(new_cols)} column(s) from {filename}")
            for col in new_cols:
                matched_count = merged_df[col].notna().sum()
                print(f"    {col}: {matched_count}/{len(merged_df)} matches ({100*matched_count/len(merged_df):.1f}%)")

        except Exception as e:
            print(f"Error processing {filename}: {str(e)}")
            continue

    # Convert DataSyncMarker_channel_1 from UNIX time to Date and Time columns
    if 'DataSyncMarker_channel_1' in merged_df.columns:
        print(f"\nCalculating Date and Time columns using DataSyncMarker and EmotiBit timestamps...")
        valid_sync_timestamps = merged_df['DataSyncMarker_channel_1'].notna()

        if valid_sync_timestamps.any():
            try:
                first_sync_idx = merged_df[valid_sync_timestamps].index[0]
                first_sync_unix = merged_df.loc[first_sync_idx, 'DataSyncMarker_channel_1']
                sync_emotibit_time = merged_df.loc[first_sync_idx, 'timestamp']

                if 1577836800 <= first_sync_unix <= 1893456000:
                    import pytz
                    utc_datetime = pd.to_datetime(first_sync_unix, unit='s', utc=True)
                    est_tz = pytz.timezone('US/Eastern')
                    initial_datetime_est = utc_datetime.tz_convert(est_tz)

                    print(f"Initial sync point at row {first_sync_idx}:")
                    print(f"  UNIX timestamp: {first_sync_unix}")
                    print(f"  EST time: {initial_datetime_est}")
                    print(f"  Corresponding EmotiBit timestamp: {sync_emotibit_time}")

                    time_diffs = merged_df['timestamp'] - sync_emotibit_time
                    calculated_datetimes = initial_datetime_est.tz_localize(None) + pd.to_timedelta(time_diffs, unit='s')

                    date_col = calculated_datetimes.dt.strftime('%Y-%m-%d')
                    time_col = calculated_datetimes.dt.strftime('%H:%M:%S.%f').str[:-3]

                    columns = list(merged_df.columns)
                    if 'Condition' in columns:
                        insert_pos = columns.index('Condition') + 1
                    elif 'ID' in columns:
                        insert_pos = columns.index('ID') + 1
                    else:
                        insert_pos = 0

                    columns.insert(insert_pos, 'Date')
                    columns.insert(insert_pos + 1, 'Time')
                    merged_df['Date'] = date_col
                    merged_df['Time'] = time_col
                    merged_df = merged_df[columns]

                    print(f"Successfully created Date and Time columns for all {len(merged_df)} rows (EST timezone)")
                    print(f"Duration: {(calculated_datetimes.max() - calculated_datetimes.min()).total_seconds():.2f} seconds")

                    start_idx = max(0, first_sync_idx - 2)
                    end_idx = min(len(merged_df), first_sync_idx + 3)
                    print(f"\nExample rows around sync point (row {first_sync_idx}):")
                    for i in range(start_idx, end_idx):
                        marker = " <- SYNC POINT" if i == first_sync_idx else ""
                        time_diff = merged_df.loc[i, 'timestamp'] - sync_emotibit_time
                        print(f"  Row {i}: EmotiBit={merged_df.loc[i, 'timestamp']:.3f}, diff={time_diff:+.3f}s, Time={time_col.iloc[i]}{marker}")
                else:
                    print(f"First DataSyncMarker value {first_sync_unix} is outside expected range (2020-2030)")

            except Exception as e:
                print(f"Error calculating Date and Time columns: {str(e)}")
        else:
            print("No valid DataSyncMarker_channel_1 values found")

    # Apply forward fill (excludes keyboard columns)
    if use_ffill:
        print(f"\nApplying forward fill to missing values...")
        missing_before = merged_df.isnull().sum().sum()
        keyboard_cols = [col for col in merged_df.columns if 'Keyboard' in col]
        exclude_cols = ['timestamp', 'ID', 'Condition', 'Date', 'Time'] + keyboard_cols
        data_columns = [col for col in merged_df.columns if col not in exclude_cols]
        merged_df[data_columns] = merged_df[data_columns].ffill()
        missing_after = merged_df.isnull().sum().sum()
        print(f"Missing values before ffill: {missing_before}")
        print(f"Missing values after ffill: {missing_after}")
        print(f"Filled {missing_before - missing_after} missing values")
        if keyboard_cols:
            print(f"Note: Keyboard columns excluded from forward fill: {keyboard_cols}")

    # Rename columns:
    #   - strip _value suffix
    #   - TEMP1_X -> TEMP_X
    #   - _I suffix -> _IS
    #   - Keyboard -> keyboard_C, Keyboard_IS -> keyboard_IS
    rename_map = {}
    for col in merged_df.columns:
        new_col = col
        if new_col.endswith('_value'):
            new_col = new_col[:-6]
        new_col = new_col.replace('TEMP1_', 'TEMP_')
        if new_col.endswith('_I'):
            new_col = new_col + 'S'
        if new_col == 'Keyboard':
            new_col = 'keyboard_C'
        elif new_col == 'Keyboard_IS':
            new_col = 'keyboard_IS'
        if new_col != col:
            rename_map[col] = new_col
    merged_df = merged_df.rename(columns=rename_map)
    print(f"\nRenamed {len(rename_map)} columns: {rename_map}")

    # Reorder columns
    desired_order = [
        'ID', 'Condition', 'Date', 'Time', 'timestamp',
        'DataSyncMarker', 'DataSyncMarker_channel_1', 'Reliability',
        'PPG_GRN_C', 'PPG_RED_C', 'PPG_IR_C', 'EDA_C', 'HR_C', 'TEMP_C', 'THERM_C', 'keyboard_C',
        'Neon Companion_Neon Gaze_PupilDiameter',
        'PPG_GRN_IS', 'PPG_RED_IS', 'PPG_IR_IS', 'EDA_IS', 'HR_IS', 'TEMP_IS', 'THERM_IS', 'keyboard_IS'
    ]
    ordered_cols = [col for col in desired_order if col in merged_df.columns]
    remaining_cols = [col for col in merged_df.columns if col not in desired_order]
    if remaining_cols:
        print(f"Note: extra columns appended at end: {remaining_cols}")
    merged_df = merged_df[ordered_cols + remaining_cols]

    merged_df.to_csv(output_path, index=False)

    print(f"\n=== MERGE SUMMARY ===")
    print(f"Final dataset: {len(merged_df)} rows, {len(merged_df.columns)} columns")
    print(f"Columns: {list(merged_df.columns)}")
    print(f"Time range: {merged_df['timestamp'].min():.3f} to {merged_df['timestamp'].max():.3f}")
    print(f"Duration: {merged_df['timestamp'].max() - merged_df['timestamp'].min():.2f} seconds")
    if participant_id is not None:
        print(f"Participant ID: {participant_id}")
    if condition is not None:
        print(f"Condition: {condition}")
    if video_condition is not None:
        print(f"Video condition: {video_condition}")

    print(f"\n=== FINAL DATA COMPLETENESS ===")
    for col in merged_df.columns:
        if col not in ['timestamp', 'ID', 'Condition', 'Date', 'Time']:
            non_null_count = merged_df[col].notna().sum()
            completeness = (non_null_count / len(merged_df)) * 100
            print(f"{col}: {non_null_count}/{len(merged_df)} ({completeness:.1f}% complete)")

    return merged_df

In [44]:
def _label_video_condition(binary_sequence, video_condition):
    """
    Label binary sequence based on video condition and playback order.
    
    Parameters:
    - binary_sequence: Series of 1s and 2s (1=video playback, 2=no video)
    - video_condition: Integer 1-6 indicating which of the 6 video orders from ExperimentPlayer
    
    Returns:
    - Series with labels: "Unreliable", "No Video", "Reliable", or original binary values
    """
    
    # Define the 6 video orders from MHMUAV Experiment Player.py
    video_orders = {
        1: ('unreliable', 'reliable', 'mix'),
        2: ('unreliable', 'mix', 'reliable'),
        3: ('reliable', 'unreliable', 'mix'),
        4: ('reliable', 'mix', 'unreliable'),
        5: ('mix', 'unreliable', 'reliable'),
        6: ('mix', 'reliable', 'unreliable')
    }
    
    # For conditions other than 1, return original binary values
    if video_condition != 1:
        return binary_sequence
    
    # For condition 1: ('unreliable', 'reliable', 'mix')
    # Need to label the sequence [1,1,1,2,2,2,1,1,1] as:
    # - All 1s BEFORE first sequence of 2s → "Unreliable" 
    # - All 2s → "No Video"
    # - All 1s AFTER first sequence of 2s → "Reliable"
    
    # Convert to numpy array for easier manipulation
    seq_array = binary_sequence.values
    labels = seq_array.astype(object)  # Create array that can hold strings
    
    # Find the first occurrence where value changes from 1 to 2 (start of first no-video period)
    first_no_video_start = None
    
    for i in range(len(seq_array)):
        if seq_array[i] == 2:
            first_no_video_start = i
            break
    
    # Apply labels with forward fill logic
    current_label = None
    
    for i in range(len(seq_array)):
        if seq_array[i] == 2:
            # All 2s are "No Video"
            current_label = "No Video"
            labels[i] = current_label
        elif seq_array[i] == 1:
            # Determine label for 1s based on position relative to first no-video period
            if first_no_video_start is None or i < first_no_video_start:
                # 1s before first sequence of 2s
                current_label = "Unreliable"
            else:
                # 1s after first sequence of 2s
                current_label = "Reliable"
            labels[i] = current_label
        else:
            # For any other values (like NaN), forward fill the current label
            if current_label is not None:
                labels[i] = current_label
    
    return pd.Series(labels, index=binary_sequence.index)

In [45]:
def extract_is_keyboard_csv(is_xdf_path, cmd_streams, output_dir):
    """
    Extract the Keyboard stream from the Intelligence Support XDF file,
    convert its timestamps to the Commander LSL timeline using the shared
    DataSyncMarker UNIX anchor, and save as Keyboard_IS.csv in output_dir.

    Parameters:
    - is_xdf_path: Path to the IS XDF file
    - cmd_streams: Commander streams list (from pyxdf.load_xdf on Commander XDF)
    - output_dir: Directory to save Keyboard_IS.csv (same dir as other exported CSVs)

    Returns:
    - Path to the saved Keyboard_IS.csv, or None if extraction failed
    """
    # --- Find Commander sync anchor (DataSyncMarker channel_1 UNIX timestamp) ---
    cmd_sync_lsl = None
    cmd_sync_unix = None
    for stream in cmd_streams:
        if stream['info']['name'][0] == 'DataSyncMarker':
            ts = stream['time_stamps']
            data = stream['time_series']
            if isinstance(data, np.ndarray) and data.ndim > 1:
                for i, unix_val in enumerate(data[:, 1]):
                    if 1577836800 <= unix_val <= 1893456000:
                        cmd_sync_lsl = ts[i]
                        cmd_sync_unix = unix_val
                        break
            break

    if cmd_sync_lsl is None:
        print("ERROR: No valid DataSyncMarker sync point found in Commander streams")
        return None
    print(f"Commander sync anchor: LSL={cmd_sync_lsl:.6f}, UNIX={cmd_sync_unix:.6f}")

    # --- Load IS XDF and find IS sync anchor + Keyboard stream ---
    print(f"Loading IS XDF file: {is_xdf_path}")
    is_streams, _ = pyxdf.load_xdf(is_xdf_path)

    is_keyboard = None
    is_sync_lsl = None
    is_sync_unix = None

    for stream in is_streams:
        name = stream['info']['name'][0]
        if name == 'Keyboard':
            is_keyboard = stream
            print(f"  Found IS Keyboard stream: {len(stream['time_stamps'])} events")
        elif name == 'DataSyncMarker':
            ts = stream['time_stamps']
            data = stream['time_series']
            if isinstance(data, np.ndarray) and data.ndim > 1:
                for i, unix_val in enumerate(data[:, 1]):
                    if 1577836800 <= unix_val <= 1893456000:
                        is_sync_lsl = ts[i]
                        is_sync_unix = unix_val
                        break

    if is_keyboard is None:
        print("ERROR: No Keyboard stream found in IS XDF file")
        return None
    if is_sync_lsl is None:
        print("ERROR: No valid DataSyncMarker sync point found in IS XDF file")
        return None

    print(f"IS sync anchor:        LSL={is_sync_lsl:.6f}, UNIX={is_sync_unix:.6f}")

    unix_diff = abs(cmd_sync_unix - is_sync_unix)
    if unix_diff > 1.0:
        print(f"WARNING: UNIX timestamps differ by {unix_diff:.3f}s — sync may be inaccurate")
    else:
        print(f"UNIX timestamps match (diff={unix_diff:.6f}s) — sync OK")

    # --- Convert IS keyboard timestamps to Commander LSL timeline ---
    # Both devices recorded the same real-world UNIX event, so:
    #   cmd_lsl = cmd_sync_lsl + (is_lsl - is_sync_lsl)
    is_kb_ts = np.array(is_keyboard['time_stamps'])
    is_kb_data = is_keyboard['time_series']

    if isinstance(is_kb_data, np.ndarray) and is_kb_data.ndim > 1:
        is_kb_data = is_kb_data.flatten()

    cmd_equivalent_ts = cmd_sync_lsl + (is_kb_ts - is_sync_lsl)

    df = pd.DataFrame({
        'timestamp': cmd_equivalent_ts,
        'value': is_kb_data
    })

    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, 'Keyboard_IS.csv')
    df.to_csv(output_path, index=False)
    print(f"Saved IS Keyboard to {output_path} ({len(df)} events)")
    return output_path

In [46]:
# Load the Commander XDF file
file_path = "/Users/debbiehsu/Downloads/23_C.xdf"  # Commander XDF file path
streams, header = pyxdf.load_xdf(file_path)

Stream 1: Calculated effective sampling rate 146.2071 Hz is different from specified rate 200.0000 Hz.


In [47]:
# Check streams
for idx, stream in enumerate(streams):
    print(f"Stream Index: {idx}")
    print(f"Source ID: {stream['info'].get('source_id', ['Unknown'])[0]}")
    print(f"Stream Name: {stream['info']['name'][0]}")
    print(f"Channel Count: {stream['info']['channel_count'][0]}")
    print(f"Sampling Rate: {stream['info']['nominal_srate'][0]} Hz\n")

Stream Index: 0
Source ID: MD-V5-0001023
Stream Name: THERM
Channel Count: 1
Sampling Rate: 7.500000000000000 Hz

Stream Index: 1
Source ID: MD-V6-0000405
Stream Name: THERM
Channel Count: 1
Sampling Rate: 7.500000000000000 Hz

Stream Index: 2
Source ID: MD-V5-0001023
Stream Name: TEMP1
Channel Count: 1
Sampling Rate: 7.500000000000000 Hz

Stream Index: 3
Source ID: None
Stream Name: Keyboard
Channel Count: 1
Sampling Rate: 0.000000000000000 Hz

Stream Index: 4
Source ID: MD-V6-0000405
Stream Name: PPG_IR
Channel Count: 1
Sampling Rate: 25.00000000000000 Hz

Stream Index: 5
Source ID: MD-V5-0001023
Stream Name: PPG_RED
Channel Count: 1
Sampling Rate: 25.00000000000000 Hz

Stream Index: 6
Source ID: MD-V6-0000405
Stream Name: PPG_GRN
Channel Count: 1
Sampling Rate: 25.00000000000000 Hz

Stream Index: 7
Source ID: MD-V6-0000405
Stream Name: TEMP1
Channel Count: 1
Sampling Rate: 7.500000000000000 Hz

Stream Index: 8
Source ID: MD-V5-0001023
Stream Name: PPG_IR
Channel Count: 1
Sampling Ra

In [48]:
# Path to Intelligence Support XDF file
is_xdf_path = "/Users/debbiehsu/Downloads/23_I.xdf"  # Replace with your actual IS XDF file path

# Export Commander streams to CSV files
exported_files = export_streams_to_csv(streams, output_dir="23_Data_C")

# Extract IS Keyboard stream, convert to Commander timeline, save to 23_Data_C/Keyboard_IS.csv
extract_is_keyboard_csv(is_xdf_path, streams, output_dir="23_Data_C")

# Merge all CSV files with participant data
merge_csv_files(
    input_dir="23_Data_C",
    output_path="merged_data_23_C.csv",
    use_ffill=True,
    tolerance=0.002,  # 10ms tolerance for temporal alignment (seconds unit: 0.001 = 1ms, 0.01 = 10ms)
    participant_id=23,  # Set participant ID
    condition="Low",       # Set condition (e.g., "High", "Low")
    video_condition=4  # Set video condition (1-6)
)

Exported stream 0 (THERM) to 23_Data_C/THERM_C.csv
  Source ID: MD-V5-0001023
  Data shape: (21055, 1)
  Sample count: 21055

Exported stream 1 (THERM) to 23_Data_C/THERM_I.csv
  Source ID: MD-V6-0000405
  Data shape: (21055, 1)
  Sample count: 21055

Exported stream 2 (TEMP1) to 23_Data_C/TEMP1_C.csv
  Source ID: MD-V5-0001023
  Data shape: (21054, 1)
  Sample count: 21054

Exported stream 3 (Keyboard) to 23_Data_C/Keyboard.csv
  Source ID: Unknown
  Data shape: (232, 1)
  Sample count: 232

Exported stream 4 (PPG_IR) to 23_Data_C/PPG_IR_I.csv
  Source ID: MD-V6-0000405
  Data shape: (69847, 1)
  Sample count: 69847

Exported stream 5 (PPG_RED) to 23_Data_C/PPG_RED_C.csv
  Source ID: MD-V5-0001023
  Data shape: (69988, 1)
  Sample count: 69988

Exported stream 6 (PPG_GRN) to 23_Data_C/PPG_GRN_I.csv
  Source ID: MD-V6-0000405
  Data shape: (69847, 1)
  Sample count: 69847

Exported stream 7 (TEMP1) to 23_Data_C/TEMP1_I.csv
  Source ID: MD-V6-0000405
  Data shape: (21055, 1)
  Sample co

,ID,Condition,Date,Time,timestamp,DataSyncMarker,DataSyncMarker_channel_1,Reliability,PPG_GRN_C,PPG_RED_C,...,keyboard_C,Neon Companion_Neon Gaze_PupilDiameter,PPG_GRN_IS,PPG_RED_IS,PPG_IR_IS,EDA_IS,HR_IS,TEMP_IS,THERM_IS,keyboard_IS
0,23,Low,2025-08-28,10:23:14.811,160571.037,NaN,NaN,NaN,7839.0,130854.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,23,Low,2025-08-28,10:23:14.852,160571.078,NaN,NaN,NaN,7833.0,130831.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,23,Low,2025-08-28,10:23:14.892,160571.118,NaN,NaN,NaN,7810.0,130825.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,23,Low,2025-08-28,10:23:14.932,160571.158,NaN,NaN,NaN,7809.0,130826.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,23,Low,2025-08-28,10:23:14.972,160571.198,NaN,NaN,NaN,7788.0,130810.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70389,23,Low,2025-08-28,11:10:01.779,163378.005,2.0,1.756394e+09,No Video,12731.0,139785.0,...,NaN,4.168324,3019.0,76091.0,139739.0,0.030147,51.51,38.333,34.842,NaN
70390,23,Low,2025-08-28,11:10:01.819,163378.045,2.0,1.756394e+09,No Video,12785.0,140013.0,...,NaN,4.168324,3019.0,76091.0,139739.0,0.030147,51.51,38.333,34.842,NaN
70391,23,Low,2025-08-28,11:10:01.859,163378.085,2.0,1.756394e+09,No Video,12821.0,140164.0,...,NaN,4.168324,3019.0,76091.0,139739.0,0.030147,51.51,38.333,34.842,NaN
70392,23,Low,2025-08-28,11:10:01.899,163378.125,2.0,1.756394e+09,No Video,12844.0,140178.0,...,NaN,4.168324,3019.0,76091.0,139739.0,0.030147,51.51,38.333,34.842,NaN
